# CPU baselines, architectural controls and loss comparisons
Run cells in order. Notebook outputs remain visible; durable artifacts are written to results/v2.


In [ ]:
from pathlib import Path
import os, sys, json, errno, time
# VS Code + Colab: load the synchronized source folder, without browser uploads.
DRIVE = Path("/content/drive/MyDrive")
try:
    from google.colab import drive
except ImportError:
    drive = None
def with_drive_retry(operation, attempts=3, reconnect=None):
    for attempt in range(attempts):
        try:
            return operation()
        except OSError as exc:
            disconnected = exc.errno in {errno.ENOTCONN, errno.EIO, errno.ESTALE} or "Transport endpoint" in str(exc)
            if not disconnected or (drive is None and reconnect is None) or attempt == attempts-1:
                raise
            print(f"Drive disconnected. Reconnecting and retrying ({attempt+1}/{attempts-1})...", flush=True)
            if reconnect is not None:
                reconnect()
            else:
                drive.mount("/content/drive", force_remount=True)
            time.sleep(1)

def ensure_drive():
    if drive is not None:
        if not DRIVE.is_dir():
            drive.mount("/content/drive", force_remount=False)
        # Probe the filesystem: an existing mount directory can still be disconnected.
        next(DRIVE.iterdir(), None)
with_drive_retry(ensure_drive)

def resolve_v2(bases):
    checked = []
    for base in bases:
        for candidate in (base, base/"implementation/v2", base/"v2"):
            checked.append(str(candidate))
            required = ["src/atlas_v2/__init__.py", "src/atlas_v2/data.py",
                        "src/atlas_v2/models.py", "src/atlas_v2/experiments.py", "requirements.txt"]
            if all((candidate/name).is_file() for name in required):
                return candidate.resolve()
    raise FileNotFoundError(
        "The v2 Python source is not visible to this kernel. "
        "Expected the synchronized implementation/v2 folder on mounted Drive. "
        "Set ATLAS_V2_ROOT to that folder if your Drive location differs. Checked: "
        + "; ".join(checked))

manual = os.environ.get("ATLAS_V2_ROOT")
bases = [Path(manual).expanduser()] if manual else []
bases += [DRIVE/"projects/implementation/ATLAS-IDNet/implementation/v2",
          DRIVE/"projects/ATLAS-IDNet", DRIVE/"ATLAS-IDNet",
          Path.cwd(), *Path.cwd().parents, Path("/content/ATLAS-IDNet")]
V2 = with_drive_retry(lambda: resolve_v2(bases))
sys.path.insert(0, str(V2/"src"))
for name in list(sys.modules):
    if name == "atlas_v2" or name.startswith("atlas_v2."):
        del sys.modules[name]
import atlas_v2
assert Path(atlas_v2.__file__).resolve().parent == (V2/"src/atlas_v2").resolve()
print("Code:", V2)
print("Source import OK. No file upload or model weights required.")


In [ ]:
import subprocess
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(V2/"requirements.txt")])
try:
    from google.colab import drive
except ImportError:
    drive = None
if drive is not None:
    drive.mount("/content/drive")
import torch, pandas, sklearn
print("CPU runtime:", torch.__version__, "threads:", os.cpu_count())


In [ ]:
# Set this explicitly if your results directory differs.
default = "/content/drive/MyDrive/projects/implementation/ATLAS-IDNet/results/v2/experiment_config.json"
CONFIG_PATH = Path(os.environ.get("ATLAS_V2_CONFIG", default if Path("/content").exists() else str(V2/"results/experiment_config.json")))
CONFIG = json.loads(CONFIG_PATH.read_text())
RESULTS = Path(CONFIG["results"])


In [ ]:
from atlas_v2.experiments import run
VARIANTS = ["atlas", "mlp", "single_residual", "no_residual", "mean", "concat", "random_groups",
            "random_forest", "extra_trees", "hist_gradient", "logistic"]
for name, spec in CONFIG["datasets"].items():
    meta = json.loads((Path(spec["prepared"])/"metadata.json").read_text())
    if not meta.get("full_class_evaluation_supported", False):
        raise RuntimeError(f"{name}: insufficient class coverage. Review missing_classes in metadata and increase max_rows before full experiments.")
    for variant in VARIANTS:
        for seed in CONFIG["seeds"]:
            run(spec["prepared"], RESULTS/name/"experiments", variant=variant, seed=seed, epochs=CONFIG["epochs"])
    for loss in ["balanced", "focal"]:
        for seed in CONFIG["seeds"]:
            run(spec["prepared"], RESULTS/name/"experiments", seed=seed, loss=loss, epochs=CONFIG["epochs"])
